# torch.compile vs eager: Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_compile_2026-11-03.md` (C1–C7). Eager and compiled runs share this session, so they use the same GPU instance. Total about 55 min (~2 compute units).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit --format=csv

In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg lpips 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, diffusers.__version__, transformers.__version__)

In [ ]:
%cd /content
!git -C video-dit-l4 pull || git clone https://github.com/FredWen-S/video-dit-l4.git
%cd /content/video-dit-l4

## 33 frames, 4 prompts

Eager first (~15 min), then compiled (~12 min plus compile time). If the compiled run fails with a graph-break error, add `--no-fullgraph` and note it.

In [ ]:
!python bench/profile_baseline.py --frames 33 --kernels --out results/eager_33f

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --kernels --out results/compile_33f

In [ ]:
!python bench/compare_frames.py --ref results/eager_33f --test results/compile_33f

## 81 frames, 1 prompt

Eager (~12 min), then compiled (~10 min).

In [ ]:
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --kernels --out results/eager_81f

In [ ]:
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --compile blocks --kernels --out results/compile_81f

In [ ]:
!python bench/compare_frames.py --ref results/eager_81f --test results/compile_81f

In [ ]:
# Keeps the eager 33-frame frames (for C7, checked locally against the baseline) and drops the other .npy files
!zip -qr compile_results.zip results -x 'results/compile_*/*.npy' 'results/eager_81f/*.npy' && ls -lh compile_results.zip
from google.colab import files; files.download('compile_results.zip')